# Test the L0 processor v1

In [1]:
# # For testing, don't commit
# import sys
# sys.path.insert(0, "/home/jgaucher/projects/rspy/github/rs-demo/notebooks")
# import resources.test_localhost

In [1]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=4)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
image = 2ac25f08447541f8af2b8d67baf4a269
Get existing dask cluster: '2ac25f08447541f8af2b8d67baf4a269'
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/2ac25f08447541f8af2b8d67baf4a269/status
Dask workers for 'dask-eopf' are up: 4/4


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.5.1     | 6.5.1   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [2]:
# Other imports
import glob
import json
import os.path as osp
from IPython.display import JSON

from rs_common.prefect_utils import *

In [3]:
%%bash
# Install pip packages
pip install zarr vizarr


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


## Read the tasktables

Documentation: https://cpm.pages.eopf.copernicus.eu/eopf-cpm/main/processor-orchestration-guide/tasktables.html#tasktables

<div class="alert alert-block alert-warning">

Note: for now the L0 processor returns dummy values that are not usable.
</div>

In [4]:
for process_id in "s1_l0", "s3_l0":
    tasktable: dict = dpr_client.get_process(process_id)
    print(f"Tasktable for {process_id!r}:")
    display(JSON(tasktable))

Tasktable for 's1_l0':


<IPython.core.display.JSON object>

Tasktable for 's3_l0':


<IPython.core.display.JSON object>

<div class="alert alert-block alert-warning">

To be discussed: should the configuration and payload files for the processors be available from:

  * rs-dpr-service ?
  * rs-client-libraries ?
  * The user local disk ? (like in this demo)
</div>

## Run the L0 processor

In [5]:
# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config_dir = osp.join(s3_base, "config")
s3_output_dir = osp.join(s3_base, "output")
s3_report_dir = osp.join(s3_base, "reports")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./config", s3_config_dir)

# Data properties
s1_short = {
    "payload_subpath": "s1/iw_joborder.short.yaml",
    "s3_output_dir": f"{s3_output_dir}/s1.short",
    "s3_report_dir": f"{s3_report_dir}/s1.short",
}
s1 = {
    "payload_subpath": "s1/iw_joborder.yaml",
    "s3_output_dir": f"{s3_output_dir}/s1",
    "s3_report_dir": f"{s3_report_dir}/s1",
}
s3 = {
    "payload_subpath": "s3/s3_dordop_payload.yaml",
    "s3_output_dir": f"{s3_output_dir}/s3",
    "s3_report_dir": f"{s3_report_dir}/s3",
}

# Update local configuration files depending on the environment, 
# and upload them again to the s3 bucket.

# Update and upload secret file
await dpr_client.update_configuration(
    local_path = "./config/secrets.json",
    s3_path = osp.join(s3_config_dir, "secrets.json"),
)

# For each data...
for data in s1_short, s1, s3:

    # Remove existing output and report folders
    this_s3_output_dir = data["s3_output_dir"]
    this_s3_report_dir = data["s3_report_dir"]
    print(f"Remove existing zarr products from: {this_s3_output_dir!r}")

    # TODO: I have strange errors when deleting: 
    # An error occurred (SignatureDoesNotMatch) when calling the DeleteObjects operation: 
    # The request signature we calculated does not match the signature you provided. 
    # Check your key and signing method.
    try:
        s3_delete(this_s3_output_dir)
        s3_delete(this_s3_report_dir)
    except Exception as e:
        print(e)
    
    # Update local configuration file depending on the environment, upload it to the s3 bucket,
    # and initialize output bucket folders.
    await dpr_client.update_configuration(
        local_path = osp.join("./config", data["payload_subpath"]),
        s3_path = osp.join(s3_config_dir, data["payload_subpath"]),
        is_payload = True,
        # Specific environment variables to expand in the payload file
        PREFECT_BUCKET_NAME=os.environ["PREFECT_BUCKET_NAME"], 
        OUTPUT_DIR=data["s3_output_dir"],
    )

16:11:19.893 | INFO    | prefect.S3Bucket - Uploading from 'config/secrets.json' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/secrets.json'.

16:11:19.895 | INFO    | prefect.S3Bucket - Uploading from 'config/l0_dask_configuration.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/l0_dask_configuration.yaml'.

16:11:19.897 | INFO    | prefect.S3Bucket - Uploading from 'config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/logging_config.yaml'.

16:11:19.898 | INFO    | prefect.S3Bucket - Uploading from 'config/s1/iw_joborder.short.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

16:11:19.899 | INFO    | prefect.S3Bucket - Uploading from 'config/s1/iw_configuration.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_configuration.yaml'.

...

16:11:20.742 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmphzlc_68u' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/output/s1/S1A_20240410083700053369/.empty'.

16:11:20.753 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpyof1nmul' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_joborder.yaml'.

Remove existing zarr products from: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s3'
An error occurred (SignatureDoesNotMatch) when calling the DeleteObjects operation: The request signature we calculated does not match the signature you provided. Check your key and signing method.


16:11:20.966 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpqxvzi6yk' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/output/s3/S3A_20250612034536048530/.empty'.

16:11:20.976 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpr96to7qc' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'.

### S1 short data

In [9]:
%%time
this_s3_output_dir = s1_short["s3_output_dir"]
this_s3_report_dir = s1_short["s3_report_dir"]

# Run processor
result = dpr_client.run_process(
    "s1_l0",
    s3_config_dir = s3_config_dir,
    payload_subpath = s1_short["payload_subpath"],
    s3_report_dir = this_s3_report_dir,
)
dpr_client.wait_for_job(result, logger=logger, poll_interval=5)

16:19:23.138 [INFO] (resources.utils) job_status: {'processID': 'dpr-service', 'progress': 50, 'type': 'process', 'created': '2025-06-26T16:19:14Z', 'started': '2025-06-26T16:19:14Z', 'message': 'In progress', 'status': 'running', 'updated': '2025-06-26T16:19:23Z', 'jobID': '9d9ba948-955f-49f1-9455-903c7205986d'}
16:19:23.139 [INFO] (resources.utils) -----  job '9d9ba948-955f-49f1-9455-903c7205986d': RUNNING 

16:19:28.158 [INFO] (resources.utils) job_status: {'processID': 'dpr-service', 'progress': 50, 'type': 'process', 'created': '2025-06-26T16:19:14Z', 'started': '2025-06-26T16:19:14Z', 'message': 'In progress', 'status': 'running', 'updated': '2025-06-26T16:19:23Z', 'jobID': '9d9ba948-955f-49f1-9455-903c7205986d'}
16:19:28.159 [INFO] (resources.utils) -----  job '9d9ba948-955f-49f1-9455-903c7205986d': RUNNING 

16:19:33.182 [INFO] (resources.utils) job_status: {'processID': 'dpr-service', 'progress': 100, 'type': 'process', 'created': '2025-06-26T16:19:14Z', 'started': '2025-06-26

CPU times: user 54.8 ms, sys: 19.6 ms, total: 74.4 ms
Wall time: 18.7 s


{}

In [13]:
# Download reports folder from the s3 bucket
local_report_dir = f"./reports/s1.short"
await s3_download_dir(this_s3_report_dir, local_report_dir)

# Display logs here
log_file = glob.glob(osp.join(local_report_dir, "**/*.processor.log"), recursive=True)[0]
with open(log_file, "r", encoding="utf-8") as openend:
    print(f"Log file {log_file!r}:\n{openend.read()}")

Log file './reports/s1.short/iw_joborder.short.log':
INFO:eopf.trigger.local:RUN with {'general_configuration': {'logging': {'level': 'INFO'}, 'triggering__use_basic_logging': True, 'triggering__use_default_filename': False, 'triggering__wait_before_exit': 0, 'dask__export_graphs': './reports/graphs'}, 'workflow': [{'name': 's1_l0_processor', 'active': True, 'module': 'l0.s1.s1_l0_processor', 'processing_unit': 'S1L0Processor', 'inputs': {'CADUS': 'S1ACADU'}, 'outputs': {'datatakeid_.*': 'output_folder'}, 'adfs': {'osf': 'OSF', 'fro': 'FRO'}, 'parameters': {'acquisition_report_output_path': './output_params/s1', 'streaming_mode': False, 'temporary_path': './output_params/tmp'}}], 'I/O': {'input_products': [{'id': 'S1ACADU', 'path': 's3://rs-dev-cluster-temp/stations/CADIP/test-l0-develop/S1A_20250611050700059594.short', 'store_type': 'cadu', 'store_params': {'storage_options': {'key': ***}', 'secret': ***}', 'client_kwargs': {'endpoint_url': ***}', 'region_name': ***}'}}}}], 'adfs': [{

In [6]:
# Download output data from the s3 bucket
print(f"Output products generated on: {this_s3_output_dir!r}")
local_output_dir = f"./outputs/s1.short"
#await s3_download_dir(this_s3_output_dir, local_output_dir)

Output products generated on: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s3'


In [11]:
%%bash 
pip install -U jupyterlab ipywidgets jupyterlab-widgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 47.2 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: jupyterlab-server
    Found existing installation: jupyterlab_server 2.25.0
    Uninstalling jupyterlab_server-2.25.0:
      Successfully uninstalled jupyterlab_server-2.25.0
  Attempting uninstall: jupyterlab
    Found existing installation: jupyterlab 4.0.7
    Uninstalling jupyterlab-4.0.7:
      Successfully uninstalled jupyterlab-4.0.7



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:

import vizarr
import zarr

local_zarr = glob.glob(osp.join(local_output_dir, "**/*.zarr"), recursive=True)[0]
print(local_zarr)
store = zarr.open(local_zarr)
viewer = vizarr.Viewer()
viewer.add_image(store)
viewer

./output/s1.short/S1A_20240410083700053369.short/S01IWRAW__20250611T050659_0008_A102_T282_76618_DV.zarr


Viewer()